In [3]:
import json
import pandas
import os
import re

In [4]:
%cd "E:/src code 2/python 2/KG"
from src.index.entity_extractor import EntityExtractor
from src.utils.config_loader import ConfigLoader
from src.llm.gemini import Gemini_LLM
from src.utils.utils import read_file
from src.preprocess.utils import remove_parentheses_content
import os
config = ConfigLoader().get_config_from_file(r"E:\src code 2\python 2\Legal_RAG\config\config.yaml")
llm = Gemini_LLM(config=config)

entities = ["Programming Language", "Library", "Software", "Technology", "Task", "Country", "City", "District"]


E:\src code 2\python 2\KG


# Dữ liệu dạng json

In [12]:
def load_json(file):
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

def extract_relations(data, parent=None, relations=None):
    if relations is None:
        relations = []

    if isinstance(data, dict):
        for key, value in data.items():
            if parent is not None:
                relations.append((parent, key))
            extract_relations(value, key, relations)

    return relations

def get_abbreviation(s:str):
    if not ("(" in s and ")" in s):
        return None, None
    
    abbreviation = s.split("(")[1].split(")")[0]

    if " " in abbreviation:
        return None, None
    
    word = remove_parentheses_content(s)

    for c in """%#@!^&*:,/.-+'()\"""":
        word = word.replace(c, ' ')

    word = re.sub(r'\s+', ' ', word).strip().lower()

    abbreviation = re.sub(r'\s+', ' ', abbreviation).strip().lower()
    return abbreviation, word

In [6]:
folder = "E:/data/kg/"
with open("E:/data/kg_processed/graph.txt", 'w', encoding="utf-8") as f:
    for file in os.listdir(folder):
        if not ".json" in file: continue
        data = load_json(folder + file)
        relations = extract_relations(data)
        for relation in relations:
            f.write(f"{relation[0]}\t{relation[1]}\n")



## Abbreviation

In [13]:
folder = "E:/data/kg/"
with open("data/abbreviation/abbreviation.txt", 'w', encoding="utf-8") as f:
    for file in os.listdir(folder):
        if not ".json" in file: continue
        data = load_json(folder + file)
        relations = extract_relations(data)
        for relation in relations:
            abbreviation, word = get_abbreviation(relation[0])
            if abbreviation:
                f.write(f"{abbreviation}\t{word}\n")
            abbreviation, word = get_abbreviation(relation[1])
            if abbreviation:
                f.write(f"{abbreviation}\t{word}\n")

# Dữ liệu crawl từ web

In [39]:
folder = "E:/data/it_crawl/"
relations = []
for file in os.listdir(folder):
    if not "_processed.txt" in file: continue
    text = read_file(folder + file)
    for line in text.split("\n"):
        line = line.strip().strip('(').strip(')')
        if line.startswith('"relationship"<|>'):
            _, e1, e2, *arg = line.split("<|>")
            relations.append((e1.strip(), e2.strip()))
relations = list(set(relations))
with open("E:/data/kg_processed/crawl.txt", 'w', encoding="utf-8") as f:
    for (e1, e2) in relations:
        if e1.strip().lower() == "software" or e1.strip().lower() == "technology": continue
        f.write(f"{e1}\t{e2}\n")


In [ ]:
folder = "E:/data/it_crawl_gemini/"
relations = []
for file in os.listdir(folder):
    if not "_processed.txt" in file: continue
    text = read_file(folder + file)
    for line in text.split("\n"):
        line = line.strip().strip('(').strip(')')
        if line.startswith('"relationship"<|>'):
            _, e1, e2, *arg = line.split("<|>")
            relations.append((e1.strip(), e2.strip()))
relations = list(set(relations))
with open("E:/data/kg_processed/crawl_gemini.txt", 'w', encoding="utf-8") as f:
    for (e1, e2) in relations:
        if e1.strip().lower() == "software" or e1.strip().lower() == "technology": continue
        f.write(f"{e1}\t{e2}\n")